# Credit Risk Prediction Model
## Notebook 1: Business Problem and Lending Decision Framework

### Project Objective

The goal of this project is to develop a machine-learning-assisted credit
decision system for consumer lending.

Rather than treating credit risk purely as a classification problem, this
project approaches lending as a business decision under uncertainty.

For each loan application, a lender must decide whether extending credit is
worth the financial risk. A useful model must therefore do more than predict
whether an applicant is likely to default. Its predictions must support
decisions that balance:

- expected repayment revenue,
- expected losses from default,
- the opportunity cost of rejecting reliable borrowers,
- the lender's risk tolerance,
- and operational constraints.

The machine learning task will therefore be embedded inside a broader
lending decision framework.

## 1. Business Problem

A lender receives applications from potential borrowers and must decide
whether to approve or reject each application.

Approving every applicant would maximize the number of loans originated,
but would also expose the lender to potentially large credit losses.

Rejecting too many applicants would reduce credit losses, but it would also
cause the lender to lose profitable customers who would have successfully
repaid their loans.

The business problem is therefore not simply:

> "Which borrowers will default?"

Instead, the decision problem is:

> "Given the estimated credit risk of an applicant, should the lender extend
> credit to this applicant?"

This distinction is important.

Machine learning produces information about risk. The lender must then
translate that information into an actual lending decision.

## 2. Prediction Problem vs. Decision Problem

The project contains two related but distinct problems.

### Prediction problem

Estimate the probability that a borrower will experience the target credit
event.

For applicant \(i\), the model will estimate

$$
\hat{p}_i = P(Y_i = 1 \mid X_i),
\$$

where:

- $X_i$ represents the information available about applicant \(i\),
- $Y_i = 1$ represents the adverse credit outcome,
- $Y_i = 0$ represents successful repayment,
- $\hat{p}_i\$ is the model's estimated probability of the adverse event.

### Decision problem

Use the predicted risk to choose a lending action:

$$
a_i \in \{\text{Approve}, \text{Reject}\}.
$$

The optimal decision does not necessarily correspond to a probability
threshold of 0.50.

Instead, the appropriate threshold depends on the economic consequences of
the two possible prediction errors:

1. approving a borrower who defaults;
2. rejecting a borrower who would have repaid successfully.

The prediction model therefore serves as an input to the decision system
rather than being the final decision itself.

## 3. Stakeholders

A credit decision system affects several stakeholders.

### Lender

The lender wants to originate profitable loans while controlling credit
losses and maintaining an acceptable risk profile.

### Borrower

Applicants want access to credit and expect lending decisions to be based
on relevant and reliable information.

### Risk Management Team

The risk team is responsible for controlling portfolio-level exposure and
ensuring that lending standards remain consistent with the institution's
risk appetite.

### Model / Data Science Team

The modeling team develops, validates, monitors, and explains the predictive
system used to estimate applicant risk.

### Compliance and Governance Teams

Credit models operate in a regulated environment. The organization must be
able to understand how the system behaves, monitor potential problems, and
justify the use of relevant variables and decision procedures.

These stakeholders create objectives that extend beyond predictive accuracy
alone.

## 4. Unit of Decision

The unit of analysis is an individual loan application.

For each application, the system observes applicant information available at
the time of the lending decision.

The model then estimates the applicant's credit risk.

Conceptually, the workflow is:

> Applicant Information => Credit Risk Model => Estimated Probability of Default => Lending Decision Rule => Approve / Reject => Financial Outcome

This structure prevents information observed after the lending decision from
being accidentally used to predict the original decision.

Avoiding this type of information leakage will become an important part of
the data preparation stage.

## 5. Target Outcome

The dataset provides a binary target representing whether an applicant
experienced the adverse credit outcome defined by the dataset.

We initially write

$$
Y =
\begin{cases}
1 & \text{adverse credit outcome} \\
0 & \text{no observed adverse credit outcome}
\end{cases}
$$

For readability, the project may sometimes refer to $Y=1$ as "default" or
"high-risk outcome," but the exact operational definition of the target
must be verified from the dataset documentation before modeling.

This distinction matters because a machine learning model predicts the
specific outcome encoded by the dataset, which may not be identical to every
possible business definition of loan default.

## 6. Why Classification Accuracy Is Not the Business Objective

Suppose most borrowers successfully repay their loans.

A classifier that predicts "no adverse outcome" for nearly every applicant
could achieve high accuracy while providing very little useful information
for lending decisions.

More importantly, different classification errors have different economic
consequences.

### False negative

The model classifies a genuinely risky applicant as low risk. If the loan is approved, the lender may experience a credit loss.

### False positive

The model classifies a reliable applicant as high risk.
If the applicant is rejected, the lender loses a potentially profitable
customer.

These two mistakes generally do not have equal financial consequences.
Therefore, model evaluation must eventually consider both predictive
performance and decision consequences.

## 7. Lending Interpretation of the Confusion Matrix

Assume that the model classifies applicants as either high risk or low risk.

| Actual Outcome | Predicted Low Risk | Predicted High Risk |
|---|---|---|
| Repays | True Negative | False Positive |
| Adverse outcome | False Negative | True Positive |

The corresponding business interpretations are:

### True Negative
A reliable borrower is correctly identified as low risk.

Potential action: approve the loan.

### True Positive
A risky borrower is correctly identified as high risk.

Potential action: reject the loan or apply additional review.

### False Negative
A risky borrower is incorrectly identified as low risk.

Potential consequence: the lender approves a loan that later generates a
credit loss.

### False Positive
A reliable borrower is incorrectly identified as high risk.

Potential consequence: the lender rejects a loan that could have produced
profit.

The relative cost of false negatives and false positives will later help
determine the lending threshold.

## 8. From Predicted Risk to Lending Decision

Suppose the model estimates an applicant's probability of an adverse credit
outcome as

$$
\hat{p}_i.
$$

A simple decision system could use a risk threshold $t$:

$$
\text{Decision}_i =
\begin{cases}
\text{Approve}, & \hat{p}_i < t \\
\text{Reject}, & \hat{p}_i \ge t
\end{cases}
$$

The value of $t$ controls the lender's risk tolerance.

### Lower threshold

More applicants are classified as high risk.

Expected effects:

- fewer risky loans approved,
- fewer credit losses,
- more reliable borrowers rejected,
- fewer loans originated.

### Higher threshold

More applicants are approved.

Expected effects:

- more loans originated,
- more reliable borrowers accepted,
- greater exposure to default risk.

The threshold therefore represents a business decision rather than merely
a modeling parameter.

## 9. Expected Financial Value of a Lending Decision

A theoretically stronger decision rule compares the expected financial value
of approving an applicant with the value of rejecting the application.

Let

$$
p = P(Y=1 \mid X)
$$

represent the estimated probability of the adverse credit outcome.

Suppose:

- $G$ = financial gain when the borrower successfully repays;
- $L$ = financial loss when the borrower experiences the adverse outcome.

Then the expected value of approving the loan can be written as

$$
EV(\text{Approve})
=
(1-p)G - pL.
$$

If rejection is treated as producing zero incremental loan profit,

$$
EV(\text{Reject}) = 0.
$$

The simplified economic decision is therefore

$$
\text{Approve if }
(1-p)G - pL > 0.
$$

This framework illustrates why credit risk probabilities alone are not
sufficient. The same predicted probability may lead to different decisions
depending on the financial characteristics of the loan.

## 10. Economic Interpretation of the Risk Threshold

Starting from

$$
(1-p)G - pL > 0,
$$

we can derive the condition under which the loan should be approved.

$$
G - pG - pL > 0
$$

$$
G > p(G+L)
$$

and therefore

$$
p < \frac{G}{G+L}.
$$

This produces an economically motivated threshold

$$
t^* = \frac{G}{G+L}.
$$

The result demonstrates an important principle:

> The optimal probability threshold depends on the relative gain from a
> successfully repaid loan and the loss from an adverse loan.

Consequently, there is no general reason for the optimal credit decision
threshold to equal 0.50.

## 11. Dataset Limitation: Prediction vs. Full Lending Economics

The available dataset primarily supports modeling credit risk.

However, a complete lending decision would ideally incorporate additional
economic information such as:

- interest revenue,
- loan duration,
- cost of capital,
- recovery after default,
- servicing costs,
- loan amount,
- and potentially customer lifetime value.

Some of these quantities may not be directly observed or may require
assumptions.

Therefore, the project will distinguish between:

### Risk prediction

Estimating the probability of the observed adverse credit outcome.

### Decision simulation

Studying how different probability thresholds and assumed error costs affect
lending decisions.

Any economic assumptions introduced later will be stated explicitly rather
than presented as observed facts.

## 12. Machine Learning Formulation

The predictive component of the lending system is a supervised binary
classification problem.

### Inputs

Applicant-level variables available before the credit decision.

These may include categories such as:

- demographic information,
- employment information,
- income and financial characteristics,
- requested credit characteristics,
- existing credit relationships,
- and historical credit information.

The precise feature set will be determined after inspecting the dataset.

### Output

A probability estimate

$$
\hat{p}_i = P(Y_i = 1 \mid X_i).
$$

### Candidate models

The project will emphasize tree-based methods:

1. Decision Tree
2. Random Forest
3. Gradient-Boosted Decision Trees

A simple baseline model may also be included to provide a reference point.

The final goal is not merely to select the model with the highest predictive
score, but to determine which model provides the most useful risk estimates
for lending decisions.

## 13. Evaluation Framework

Model evaluation will occur at multiple levels.

### 1. Discrimination

How effectively does the model rank risky applicants above safer applicants?

Potential measures:

- ROC-AUC
- Precision-Recall AUC
- precision
- recall

### 2. Probability quality

How reliable are the predicted probabilities?

Potential measures:

- log loss,
- Brier score,
- calibration curves.

Probability quality is particularly important because the lending decision
framework operates on estimated probabilities rather than only predicted
classes.

### 3. Threshold-dependent performance

How does performance change as the lender changes the approval threshold?

Examples include:

- false-positive rate,
- false-negative rate,
- approval rate,
- adverse outcome rate among approved applicants.

### 4. Business decision performance

Under explicitly stated economic assumptions, how would different models and
thresholds affect the lender's expected outcomes?

This final layer connects machine learning performance to the original
business objective.

## 14. Portfolio-Level Decision Tradeoff

A lending policy affects not only individual applicants but also the
composition of the lender's portfolio.

For a selected threshold, we will eventually examine quantities such as

$$
\text{Approval Rate}
=
\frac{\text{Number Approved}}
{\text{Total Applications}}
$$

and

$$
\text{Adverse Outcome Rate Among Approved}
=
\frac{\text{Adverse Outcomes Among Approved}}
{\text{Number Approved}}.
$$

As the lender becomes more conservative, the expected adverse outcome rate
among approved borrowers should generally decline, but the approval rate will
also decline.

This creates a portfolio-level risk-volume tradeoff.

Later notebooks will quantify this relationship using model predictions.

## 15. Information Available at Decision Time

The model should only use information that could reasonably have been
available when the lending decision was made.

Formally, if the decision occurs at time $t$, predictors should satisfy

$$
X_i \in \mathcal{I}(t),
$$

where $\mathcal{I}(t)$ represents the information available at decision
time.

Variables containing information generated after loan approval could create
target leakage.

A model containing such variables might appear highly accurate during
training while being unusable in an actual lending system.

Feature auditing will therefore include both statistical inspection and
business interpretation.

## 16. Modeling Principles

Throughout the project, the following principles will guide model
development.

### Principle 1: Prevent data leakage

Only information appropriate for the lending decision should enter the model.

### Principle 2: Preserve an untouched test set

Model selection and threshold development should not repeatedly use the final
test data.

### Principle 3: Handle class imbalance appropriately

Model performance should not be judged primarily by raw accuracy.

### Principle 4: Separate ranking from probability estimation

A model can rank applicants well while producing poorly calibrated
probabilities.

### Principle 5: Separate prediction from policy

The model estimates risk. A lending policy determines how that risk estimate
is converted into an action.

### Principle 6: Prefer interpretable business conclusions

Feature importance and model explanations should be interpreted as predictive
relationships, not automatically as causal relationships.